# Top 5 Modelos — Evaluación Individual
Entrena y evalúa los 5 mejores conjuntos de hiperparámetros encontrados en la búsqueda.

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
import mlflow
import mlflow.pytorch
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_experiment("Top_5_Models_MLP")

2026/06/01 17:18:06 INFO mlflow.tracking.fluent: Experiment with name 'Top_5_Models_MLP' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///c:/Users/Camila/OneDrive/Escritorio/Redes '
 'Neuronales/Skin-dataset-classification-CS2026/mlruns/4'), creation_time=1780345086622, experiment_id='4', last_update_time=1780345086622, lifecycle_stage='active', name='Top_5_Models_MLP', tags={}, trace_location=None, workspace='default'>

In [3]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)

In [4]:
# Paths
train_dir = "data/Split_smol/train"
val_dir   = "data/Split_smol/val/"

In [5]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)
        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return image, label

In [6]:
class MLPClassifier(nn.Module):
    def __init__(self, input_size=64*64*3, num_classes=9, dropout=0.2):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

Usando: cpu


In [8]:
# ============================================================
# TOP 5 HIPERPARÁMETROS — extraídos de la búsqueda de HP
# ============================================================
top5_hparams = [
    # Rank 1 — val_acc: 68.51%
    {"rank": 1, "input_size": 32, "batch_size": 16, "lr": 0.0001,
     "optimizer": "Adam", "dropout": 0.2, "HFlip": 0.0, "VFlip": 0.0, "RBContrast": 0.0},
    # Rank 2 — val_acc: 67.40%
    {"rank": 2, "input_size": 32, "batch_size": 16, "lr": 0.0001,
     "optimizer": "Adam", "dropout": 0.0, "HFlip": 0.0, "VFlip": 0.0, "RBContrast": 0.0},
    # Rank 3 — val_acc: 65.19%
    {"rank": 3, "input_size": 64, "batch_size": 16, "lr": 0.001,
     "optimizer": "Adam", "dropout": 0.0, "HFlip": 0.0, "VFlip": 0.0, "RBContrast": 0.5},
    # Rank 4 — val_acc: 64.64%
    {"rank": 4, "input_size": 32, "batch_size": 128, "lr": 0.001,
     "optimizer": "Adam", "dropout": 0.1, "HFlip": 0.0, "VFlip": 0.0, "RBContrast": 0.5},
    # Rank 5 — val_acc: 63.54%
    {"rank": 5, "input_size": 32, "batch_size": 64, "lr": 0.0001,
     "optimizer": "Adam", "dropout": 0.3, "HFlip": 0.0, "VFlip": 0.0, "RBContrast": 0.0},
]

In [9]:
def log_classification_report(model, loader, writer, step, prefix, classes):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)
    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)
    cls_report = classification_report(all_labels, all_preds, target_names=classes, zero_division=0)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

def evaluate(model, loader, writer, device, classes, criterion, epoch=None, prefix="val"):
    log_classification_report(model, loader, writer, step=epoch, prefix=prefix, classes=classes)
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)
    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)
    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)
    return avg_loss, acc

## Entrenamiento de los Top 5 modelos

In [10]:
n_epochs = 200
es_patience = 10

for hparams in top5_hparams:
    print(f"\n{'='*50}")
    print(f"Entrenando Rank {hparams['rank']} — lr={hparams['lr']}, dropout={hparams['dropout']}, input={hparams['input_size']}")
    print(f"{'='*50}")

    # Transforms
    train_transform = A.Compose([
        A.Resize(hparams["input_size"], hparams["input_size"]),
        A.HorizontalFlip(p=hparams["HFlip"]),
        A.VerticalFlip(p=hparams["VFlip"]),
        A.RandomBrightnessContrast(p=hparams["RBContrast"]),
        A.Normalize(),
        ToTensorV2()
    ])
    val_test_transform = A.Compose([
        A.Resize(hparams["input_size"], hparams["input_size"]),
        A.Normalize(),
        ToTensorV2()
    ])

    # Datasets y loaders
    train_dataset = CustomImageDataset(train_dir, transform=train_transform)
    val_dataset   = CustomImageDataset(val_dir,   transform=val_test_transform)
    train_loader  = DataLoader(train_dataset, batch_size=hparams["batch_size"], shuffle=True)
    val_loader    = DataLoader(val_dataset,   batch_size=hparams["batch_size"])
    num_classes   = len(set(train_dataset.labels))
    classes       = train_dataset.label_encoder.classes_

    # Modelo
    model = MLPClassifier(
        input_size=hparams["input_size"]**2*3,
        num_classes=num_classes,
        dropout=hparams["dropout"]
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=hparams["lr"]) \
        if hparams["optimizer"] == "Adam" \
        else optim.SGD(model.parameters(), lr=hparams["lr"])

    # TensorBoard
    writer = SummaryWriter(log_dir=f"runs/top5/rank_{hparams['rank']}")

    with mlflow.start_run(run_name=f"top5_rank{hparams['rank']}"):
        mlflow.log_params({
            "model": "MLPClassifier",
            "rank": hparams["rank"],
            "input_size": hparams["input_size"],
            "batch_size": hparams["batch_size"],
            "lr": hparams["lr"],
            "epochs": n_epochs,
            "optimizer": hparams["optimizer"],
            "dropout": hparams["dropout"],
            "HFlip": hparams["HFlip"],
            "VFlip": hparams["VFlip"],
            "RBContrast": hparams["RBContrast"],
            "loss_fn": "CrossEntropyLoss",
            "train_dir": train_dir,
            "val_dir": val_dir,
            "es_patience": es_patience,
        })

        best_val_acc = 0
        best_val_loss = 0
        best_train_acc = 0
        best_train_loss = 0
        best_epoch = 0

        for epoch in range(n_epochs):
            model.train()
            running_loss = 0.0
            correct, total = 0, 0

            for images, labels in tqdm(train_loader, desc=f"Rank {hparams['rank']} Epoch {epoch+1}/{n_epochs}"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

            train_loss = running_loss / len(train_loader)
            train_acc  = 100.0 * correct / total
            val_loss, val_acc = evaluate(model, val_loader, writer, device, classes, criterion, epoch=epoch, prefix="val")

            print(f"  Epoch {epoch+1}: Train {train_acc:.2f}% | Val {val_acc:.2f}%")

            writer.add_scalar("train/loss", train_loss, epoch)
            writer.add_scalar("train/accuracy", train_acc, epoch)

            mlflow.log_metrics({
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc
            }, step=epoch)

            if val_acc > best_val_acc:
                best_val_acc   = val_acc
                best_val_loss  = val_loss
                best_train_acc = train_acc
                best_train_loss = train_loss
                best_epoch     = epoch
                model_path = f"mlp_model_rank{hparams['rank']}.pth"
                torch.save(model.state_dict(), model_path)
                mlflow.log_artifact(model_path)
                mlflow.pytorch.log_model(model, artifact_path="pytorch_model")
            elif epoch > best_epoch + es_patience:
                print(f"  Early stopping en epoch {epoch+1}")
                break

        mlflow.log_metrics({
            "best_val_accuracy": best_val_acc,
            "best_val_loss": best_val_loss,
            "best_train_accuracy": best_train_acc,
            "best_train_loss": best_train_loss,
            "best_epoch": best_epoch
        }, step=epoch+1)

        print(f"  >>> Mejor val_acc: {best_val_acc:.2f}% en epoch {best_epoch+1}")
        writer.close()


Entrenando Rank 1 — lr=0.0001, dropout=0.2, input=32


Rank 1 Epoch 1/200: 100%|██████████| 42/42 [00:06<00:00,  6.52it/s]
2026/06/01 17:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Epoch 1: Train 25.11% | Val 35.56%


2026/06/01 17:19:32 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Rank 1 Epoch 2/200: 100%|██████████| 42/42 [00:05<00:00,  7.31it/s]
2026/06/01 17:19:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 17:19:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


  Epoch 2: Train 42.26% | Val 37.22%


...

  Epoch 32: Train 76.69% | Val 51.11%


Rank 5 Epoch 33/200: 100%|██████████| 11/11 [00:05<00:00,  2.01it/s]


  Epoch 33: Train 76.69% | Val 56.11%


Rank 5 Epoch 34/200: 100%|██████████| 11/11 [00:06<00:00,  1.77it/s]


  Epoch 34: Train 78.50% | Val 56.11%
  Early stopping en epoch 34
  >>> Mejor val_acc: 56.67% en epoch 23


## Comparación final de los Top 5

In [12]:
runs = mlflow.search_runs(
    experiment_names=["Top_5_Models_MLP"],
    filter_string="tags.mlflow.runName LIKE 'top5%'",
    order_by=["metrics.best_val_accuracy DESC"]
)
print(runs[["tags.mlflow.runName", "params.lr", "params.optimizer",
            "params.dropout", "params.batch_size", "params.input_size",
            "metrics.best_val_accuracy", "metrics.best_epoch"]].to_string())

  tags.mlflow.runName params.lr params.optimizer params.dropout params.batch_size params.input_size  metrics.best_val_accuracy  metrics.best_epoch
0          top5_rank2    0.0001             Adam            0.0                16                32                  58.333333                18.0
1          top5_rank1    0.0001             Adam            0.2                16                32                  57.777778                19.0
2          top5_rank5    0.0001             Adam            0.3                64                32                  56.666667                22.0
3          top5_rank4     0.001             Adam            0.1               128                32                  56.111111                15.0
4          top5_rank3     0.001             Adam            0.0                16                64                  55.555556                15.0


In [12]:
%load_ext tensorboard
%tensorboard --logdir runs/top5 --port 6007

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 29196), started 0:02:18 ago. (Use '!kill 29196' to kill it.)